In [1]:
import json
import os
import re
import time
from datetime import datetime, timezone

import requests

# ============================================================
# 1. КЛЮЧЕВЫЕ СЛОВА (адаптированы под GitHub)
# ============================================================

AGENT_TERMS = [
    "AutoML-Agent",
    "Agentic AutoML",
    "LLM agent AutoML",
    "Multi-Agent AutoML",
    "Autonomous Machine Learning",
    "AutoML pipeline agent",
    "LLM-based AutoML",
    "Self-improving ML agent",
    "Auto Research Agent",
    "Tree-Search LLM Agent",
]

BUILD_TRAIN_TERMS = [
    "Automated Machine Learning",
    "Neural Architecture Search",
    "Hyperparameter Optimization",
    "Automated Model Selection",
    "Automated Feature Engineering",
    "AutoDL",
    "Full-Pipeline AutoML",
    "HPO",
]

VALIDATION_TERMS = [
    "Automated Model Validation",
    "ML Testing",
    "Model Verification",
    "Fuzzing ML",
    "Metamorphic Testing",
    "Robustness Testing",
    "Compliance Verification",
]

MLOPS_TERMS = [
    "MLOps",
    "Automated ML Pipeline",
    "Model Deployment Automation",
    "CD4ML",
]

# Веса: чем специфичнее для темы — тем выше
TOPIC_WEIGHTS = {
    "automl-agent": 6.0,
    "agentic automl": 5.5,
    "llm agent automl": 5.0,
    "multi-agent automl": 5.0,
    "autonomous machine learning": 4.0,
    "automl pipeline agent": 4.0,
    "llm-based automl": 4.0,
    "automated model validation": 4.5,
    "model verification": 3.5,
    "ml testing": 3.0,
    "fuzzing ml": 2.5,
    "metamorphic testing": 2.5,
    "robustness testing": 2.5,
    "automl": 2.0,
    "neural architecture search": 2.0,
    "hyperparameter optimization": 2.0,
    "automated machine learning": 2.0,
    "mlops": 1.5,
}

# ============================================================
# 2. КОНФИГУРАЦИЯ
# ============================================================

GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")  # настоятельно рекомендуется
HEADERS = {"Accept": "application/vnd.github+json"}
if GITHUB_TOKEN:
    HEADERS["Authorization"] = f"Bearer {GITHUB_TOKEN}"

SEARCH_URL = "https://api.github.com/search/repositories"

# GitHub Search API: 30 запросов/мин с токеном, 10 без
REQUESTS_PER_MINUTE = 30 if GITHUB_TOKEN else 10
SLEEP_BETWEEN_QUERIES = 60.0 / REQUESTS_PER_MINUTE + 0.5

# ============================================================
# 3. ГЕНЕРАЦИЯ ЗАПРОСОВ
# ============================================================

def build_queries_full():
    """Все пары и узкие тройки с квалификаторами."""
    base_qualifiers = "language:python stars:>10 pushed:>2023-01-01"
    qs = []

    for a in AGENT_TERMS:
        for b in BUILD_TRAIN_TERMS:
            qs.append((
                f'"{a}" "{b}" {base_qualifiers}',
                f"{a} × {b}",
            ))
    for a in AGENT_TERMS[:6]:
        for v in VALIDATION_TERMS[:5]:
            qs.append((
                f'"{a}" "{v}" {base_qualifiers}',
                f"{a} × {v}",
            ))
    for a in AGENT_TERMS[:4]:
        qs.append((
            f'"{a}" AutoML {base_qualifiers}',
            f"{a} × AutoML (широкий)",
        ))
    for m in MLOPS_TERMS:
        qs.append((
            f'"{m}" agent {base_qualifiers}',
            f"MLOps × agent: {m}",
        ))
    return qs


def build_queries_fast():
    """Широкие запросы — минимум вызовов, максимум полноты."""
    agent_or = " OR ".join(f'"{a}"' for a in AGENT_TERMS[:6])
    return [
        (f'{agent_or} AutoML language:python stars:>20',
         "agents × AutoML (широкий)"),
        (f'"AutoML-Agent" OR "Agentic AutoML" OR "AutoML agent" language:python',
         "прямые попадания AutoML-agent"),
        (f'"{AGENT_TERMS[0]}" OR "{AGENT_TERMS[1]}" OR "{AGENT_TERMS[2]}" '
         f'validation language:python',
         "agents × validation"),
        (f'"{AGENT_TERMS[0]}" OR "{AGENT_TERMS[1]}" testing language:python',
         "agents × testing"),
        ('"automated model validation" OR "model verification" language:python stars:>5',
         "validation-репозитории"),
    ]

# ============================================================
# 4. ПОИСК С ДЕДУПЛИКАЦИЕЙ И ПАГИНАЦИЕЙ
# ============================================================

def search_repos(query, per_page=100, max_pages=10):
    """
    Возвращает список репозиториев по одному запросу.
    GitHub отдаёт максимум 1000 результатов на запрос (10 страниц × 100).
    """
    results = []
    for page in range(1, max_pages + 1):
        params = {
            "q": query,
            "per_page": min(per_page, 100),
            "page": page,
            "sort": "stars",
            "order": "desc",
        }
        resp = requests.get(SEARCH_URL, headers=HEADERS, params=params, timeout=30)

        if resp.status_code == 403:
            reset = resp.headers.get("X-RateLimit-Reset")
            print(f"   !! rate limit, reset at {reset}")
            time.sleep(60)
            continue
        if resp.status_code != 200:
            print(f"   !! HTTP {resp.status_code}: {resp.text[:200]}")
            break

        data = resp.json()
        items = data.get("items", [])
        if not items:
            break

        results.extend(items)

        # Если вернулось меньше per_page — страниц больше нет
        if len(items) < params["per_page"]:
            break

    return results


def run_queries(queries):
    found = {}  # full_name -> repo dict

    for i, (q, label) in enumerate(queries, 1):
        print(f"[{i:>2}/{len(queries)}] {label}")
        try:
            repos = search_repos(q)
            for repo in repos:
                found.setdefault(repo["full_name"], repo)
        except Exception as e:
            print(f"   !! ошибка: {e}")
        time.sleep(SLEEP_BETWEEN_QUERIES)

    print(f"\nВсего уникальных репозиториев: {len(found)}\n")
    return list(found.values())

# ============================================================
# 5. РАНЖИРОВАНИЕ
# ============================================================

def _norm(s):
    return re.sub(r"\s+", " ", (s or "").lower())


def keyword_score(repo):
    text = _norm(repo.get("name", "")) + " " + \
           _norm(repo.get("description", "")) + " " + \
           " ".join(repo.get("topics", []))

    score, matched = 0.0, []
    for term, w in TOPIC_WEIGHTS.items():
        if term in text:
            score += w
            matched.append(term)
    return score, matched


def recency_bonus(repo, half_life_days=365):
    pushed = repo.get("pushed_at")
    if not pushed:
        return 0.0
    try:
        dt = datetime.strptime(pushed, "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)
    except ValueError:
        return 0.0
    days = (datetime.now(timezone.utc) - dt).days
    return 1.5 * (0.5 ** (days / half_life_days))


def star_score(repo):
    """Логарифмическая шкала, чтобы 10k звёзд не забивали всё."""
    import math
    stars = repo.get("stargazers_count", 0)
    return math.log1p(stars) / math.log(10001)  # 0..1 для 0..10k звёзд


def rank(repos):
    if not repos:
        return []

    kw_raw = [keyword_score(r) for r in repos]
    kw_max = max(s for s, _ in kw_raw) or 1.0

    ranked = []
    for i, repo in enumerate(repos):
        ks, matched = kw_raw[i]
        kw_norm = ks / kw_max
        score = 4.0 * kw_norm + 2.5 * star_score(repo) + recency_bonus(repo)
        ranked.append((score, kw_norm, matched, repo))

    ranked.sort(key=lambda x: x[0], reverse=True)
    return ranked

# ============================================================
# 6. ВЫВОД И СОХРАНЕНИЕ
# ============================================================

def print_results(ranked, top_n=40, min_score=1.0):
    for i, (score, kw_norm, matched, r) in enumerate(ranked[:top_n], 1):
        if score < min_score:
            break
        print("=" * 80)
        print(f"#{i}  score={score:.2f}  (kw={kw_norm:.2f}, "
              f"stars={r.get('stargazers_count', 0)})")
        print("Совпало:", ", ".join(matched) or "—")
        print("Репозиторий:", r["full_name"])
        print("Описание:", (r.get("description") or "—")[:500])
        print("Язык:", r.get("language") or "—")
        print("Топики:", ", ".join(r.get("topics", [])) or "—")
        print("Создан:", r.get("created_at", "—")[:10])
        print("Последний push:", r.get("pushed_at", "—")[:10])
        print("URL:", r["html_url"])


def save_json(ranked, path="github_results.json"):
    data = [{
        "score": round(score, 4),
        "kw_score": round(kw_norm, 4),
        "matched": matched,
        "full_name": r["full_name"],
        "description": r.get("description"),
        "url": r["html_url"],
        "stars": r.get("stargazers_count", 0),
        "forks": r.get("forks_count", 0),
        "language": r.get("language"),
        "topics": r.get("topics", []),
        "created_at": r.get("created_at"),
        "pushed_at": r.get("pushed_at"),
    } for score, kw_norm, matched, r in ranked]

    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"\nСохранено в {path} ({len(data)} репозиториев)")

# ============================================================
# 7. ЗАПУСК
# ============================================================

if __name__ == "__main__":
    FAST_MODE = True   # True: ~5 запросов, ~30 сек; False: ~60 запросов, ~5 мин

    if not GITHUB_TOKEN:
        print("⚠ GITHUB_TOKEN не задан. Лимит: 10 запросов/мин, 60 запросов/час.\n"
              "  Создай токен: https://github.com/settings/tokens (scope: public_repo).\n"
              "  Затем: export GITHUB_TOKEN='ghp_...'\n")

    queries = build_queries_fast() if FAST_MODE else build_queries_full()
    print(f"Запросов: {len(queries)}\n")

    repos = run_queries(queries)
    ranked = rank(repos)
    print_results(ranked, top_n=40)
    save_json(ranked)

⚠ GITHUB_TOKEN не задан. Лимит: 10 запросов/мин, 60 запросов/час.
  Создай токен: https://github.com/settings/tokens (scope: public_repo).
  Затем: export GITHUB_TOKEN='ghp_...'

Запросов: 5

[ 1/5] agents × AutoML (широкий)
[ 2/5] прямые попадания AutoML-agent
[ 3/5] agents × validation
[ 4/5] agents × testing
   !! rate limit, reset at 1792769785
[ 5/5] validation-репозитории

Всего уникальных репозиториев: 2274

#1  score=5.69  (kw=1.00, stars=1)
Совпало: automl-agent, multi-agent automl, automl
Репозиторий: lijy-forge/Automl-Agent-Yield
Описание: 浆料屈服应力预测的多 Agent AutoML 系统（基于 AutoML-Agent）· Multi-agent AutoML with deterministic guardrails
Язык: Python
Топики: automl, langgraph, llm-agent, machine-learning, multi-agent, pinn
Создан: 2026-09-16
Последний push: 2026-10-23
URL: https://github.com/lijy-forge/Automl-Agent-Yield
#2  score=5.21  (kw=0.42, stars=1706)
Совпало: automl, automated machine learning, mlops
Репозиторий: kubeflow/katib
Описание: Automated Machine Learning on Kuber